[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/45_pretrain_data_solution.ipynb)

#  Solution: Pretraining Data Cleaning

Reference solution for pretraining data pipeline  the kind used in C4, The Pile, RedPajama, FineWeb, and other large-scale corpus processing.

```text
1. Strip whitespace
2. Filter by length (min_length, max_length)
3. Filter repetitive text (single char > 50%)
4. Deduplicate near-duplicates via 3-gram Jaccard similarity
```

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
from collections import Counter

In [ ]:
#  SOLUTION

def clean_pretrain_data(texts, min_length=50, max_length=2048,
                        dedup_threshold=0.8, filter_repetitive=True):
    """Clean pretraining text data.
    
    Inspired by:
    - C4 (Raffel et al.)  heuristic filtering pipeline
    - FineWeb (Penedo et al.)  MinHash + quality filtering
    - RedPajama-Data-v2  dedup + heuristic rules
    - The Pile (Gao et al.)  document-level dedup via MinHashLSH
    
    Uses 3-gram Jaccard similarity for near-duplicate detection
    (lightweight approximation of MinHash for small-scale use).
    """
    # 1) Strip whitespace
    cleaned = [t.strip() for t in texts]
    
    # 2) Filter by character length
    cleaned = [t for t in cleaned if min_length <= len(t) <= max_length]
    
    # 3) Filter repetitive text (single char > 50%)
    if filter_repetitive:
        result = []
        for t in cleaned:
            if len(t) == 0:
                continue
            counts = Counter(t)
            top_count = counts.most_common(1)[0][1]
            if top_count / len(t) <= 0.5:
                result.append(t)
        cleaned = result
    
    # 4) Deduplicate via 3-gram Jaccard similarity
    if dedup_threshold < 1.0 and len(cleaned) > 1:
        result = []
        for text in cleaned:
            grams = set(text[i:i+3] for i in range(len(text) - 2))
            is_dup = False
            for existing in result:
                ex_grams = set(existing[i:i+3] for i in range(len(existing) - 2))
                if grams and ex_grams:
                    jaccard = len(grams & ex_grams) / len(grams | ex_grams)
                    if jaccard > dedup_threshold:
                        is_dup = True
                        break
            if not is_dup:
                result.append(text)
        return result
    
    return cleaned

In [ ]:
#  Verify
texts = [
    "short",
    "This is a normal paragraph with enough characters to pass min_length.",
    "aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa",
    "  Another good text.  ",
    "This is a normal paragraph with enough chars to pass filter.",
]
result = clean_pretrain_data(texts, min_length=10, dedup_threshold=0.5)
print(f"{len(texts)} -> {len(result)} texts after cleaning")
for t in result:
    print(f"  {t[:60]}...")

In [ ]:
# Run judge
from torch_judge import check
check("pretrain_data")